# 1. Introduction

## 1.1 Contexte général

La biodiversité constitue un pilier du fonctionnement des écosystèmes et un enjeu central des politiques environnementales contemporaines. Sous l'effet conjugué de la destruction des habitats, de la surexploitation des ressources, de la pollution et du changement climatique, un grand nombre d'espèces animales voient leur survie compromise. L'Union internationale pour la conservation de la nature (UICN) tient un inventaire mondial de cet état de conservation à travers sa *Liste rouge*, dont l'établissement repose traditionnellement sur une expertise de terrain coûteuse et sur l'agrégation manuelle de données hétérogènes. Face à l'abondance croissante des observations écologiques disponibles, le forage de données offre une voie complémentaire : il permet d'exploiter de grands volumes de variables biologiques et environnementales afin de dégager des régularités statistiques exploitables pour anticiper le degré de menace pesant sur une espèce.

## 1.2 Problématique

La question qui structure ce projet est de savoir dans quelle mesure les caractéristiques écologiques, biologiques et géographiques d'une espèce, telles qu'elles sont consignées dans une base de données publique, portent suffisamment d'information pour qu'un modèle appris puisse prédire avec une fiabilité raisonnable la catégorie de conservation à laquelle cette espèce appartient.

## 1.3 Objectif du projet

L'objectif est de concevoir un pipeline complet de forage de données, allant de la préparation des observations jusqu'à l'évaluation comparative de modèles, afin de produire un outil capable d'attribuer automatiquement à une espèce l'une des trois classes de menace retenues : *Non menacée*, *Menacée* ou *Disparue*. Au-delà de la performance prédictive, le projet vise à mettre en pratique les notions méthodologiques du cours, en justifiant chaque choix de prétraitement, de sélection de variables, de modélisation et d'évaluation.

## 1.4 Type de problème

Le problème se formule comme une tâche d'apprentissage supervisé de classification multi-classes. Chaque observation associe un vecteur de variables explicatives, mêlant attributs taxonomiques, traits biologiques, descripteurs d'habitat et informations géographiques, à une étiquette décrivant le statut de conservation. L'apprentissage consiste à induire, à partir d'un échantillon d'entraînement, une fonction de décision capable d'assigner la bonne classe à toute nouvelle espèce. La performance de cette fonction sera mesurée sur un échantillon de test n'ayant servi ni à l'ajustement des paramètres ni à la sélection des variables, conformément aux exigences de prévention des fuites de données.

## 1.5 Plan du notebook

La suite du notebook s'organise selon la progression méthodologique du cours. Une présentation du jeu de données *World Wildlife Species Data* expose d'abord sa structure et la définition de la variable cible. Une phase d'exploration met ensuite en évidence les caractéristiques de la distribution des classes, la prévalence des valeurs manquantes et la nature des variables disponibles. La préparation des données réunit les opérations de nettoyage, d'imputation, d'encodage, de normalisation et de traitement du déséquilibre, suivie d'une étape de sélection ou de réduction de dimensions. Trois modèles de classification aux fondements distincts sont alors entraînés selon une stratégie de validation stratifiée, puis évalués à l'aide des métriques usuelles (exactitude, précision, rappel, F-mesure, matrice de confusion). Le notebook se clôt par une comparaison raisonnée des modèles et par une conclusion synthétisant les enseignements du projet.

# 2. Description des données

## 2.1 Présentation générale du jeu de données

Le projet s'appuie sur le jeu de données *World Wildlife Species Data*, mis à disposition publiquement sur la plateforme Kaggle. Celui-ci recense plus de dix mille espèces animales à travers un ensemble de variables décrivant leur identité taxonomique, leurs traits biologiques, leur répartition géographique et les pressions qui s'exercent sur leur habitat. Sa particularité tient à la diversité des informations rassemblées en une seule table : chaque ligne correspond à une espèce et chaque colonne à une dimension descriptive, ce qui en fait un support approprié à une tâche de classification supervisée à partir de variables hétérogènes.

Les attributs présents dans la base se répartissent schématiquement en trois grandes familles. Les variables taxonomiques identifient l'espèce selon la hiérarchie du vivant (règne, embranchement, classe, ordre, famille, genre). Les variables biologiques et écologiques renseignent sur les caractéristiques populationnelles et sur le type d'habitat occupé. Les variables géographiques précisent enfin l'aire de répartition et, lorsqu'elle est documentée, la localisation des menaces identifiées. À ces descripteurs s'ajoute la variable cible, dérivée de la catégorie de la *Liste rouge* de l'UICN, qui sera regroupée ultérieurement en trois classes de conservation : *Non menacée*, *Menacée* et *Disparue*.

## 2.2 Chargement et dimensions du jeu de données

Le chargement s'effectue à partir du fichier CSV placé dans le répertoire du notebook. L'affichage de la forme de la table permet de vérifier simultanément le nombre d'observations disponibles pour l'apprentissage et la dimension de l'espace de représentation, ce qui conditionne les choix méthodologiques ultérieurs en matière de sélection ou de réduction de variables.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Résolution robuste du chemin : le fichier est placé dans le dossier data/
# du dépôt, que le notebook soit ouvert depuis notebooks/ ou depuis la racine.
candidats = [
    Path('data') / 'filtered_data.csv',
    Path('..') / 'data' / 'filtered_data.csv',
]
CSV_PATH = next((p for p in candidats if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Fichier filtered_data.csv introuvable dans data/ ni ../data/."
    )

df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Fichier chargé        : {CSV_PATH}")
print(f"Nombre d'observations : {df.shape[0]}")
print(f"Nombre de variables   : {df.shape[1]}")

Fichier chargé        : ..\data\filtered_data.csv
Nombre d'observations : 159542
Nombre de variables   : 14


## 2.3 Aperçu des premières observations

Un extrait des premières lignes du tableau fournit une vision concrète du contenu des variables, de leur format et de la manière dont s'articulent les différents niveaux de description d'une espèce. Cet aperçu sert également à détecter rapidement d'éventuelles anomalies de structure, telles que des champs texte très longs, des identifiants à haute cardinalité ou des valeurs hors du domaine attendu.

In [2]:
df.head(5)

,taxonid,kingdom_name,phylum_name,class_name,order_name,family_name,genus_name,scientific_name,taxonomic_authority,infra_rank,infra_name,population,category,main_common_name
0,31665,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MYRTALES,MYRTACEAE,Myrcia,Myrcia manacalensis,Urb.,NaN,NaN,NaN,EN,Pimentillo
1,31666,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MYRTALES,MYRTACEAE,Psidium,Psidium claraense,Urb.,NaN,NaN,NaN,CR,NaN
2,31668,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MYRTALES,MYRTACEAE,Mosiera,Mosiera havanensis,(Urb.) Bisse,NaN,NaN,NaN,EN,NaN
3,31669,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MYRTALES,MELASTOMATACEAE,Henriettea,Henriettea punctata,(Griseb.) M.Gómez,NaN,NaN,NaN,CR,NaN
4,31670,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MYRTALES,MELASTOMATACEAE,Henriettea,Henriettea squamata,(Alain) Alain,NaN,NaN,NaN,LC,NaN


Les cinq premières lignes confirment l'hétérogénéité attendue du jeu de données. On y observe côte à côte des colonnes textuelles, typiquement réservées aux noms scientifiques et aux descriptions d'habitat, et des colonnes numériques associées à des comptages ou à des mesures. Plusieurs cellules apparaissent d'emblée vides, ce qui atteste de la présence de valeurs manquantes qui devront être caractérisées plus finement dans la phase suivante d'exploration.

## 2.4 Structure, types de variables et valeurs renseignées

La méthode `info()` fournit une synthèse compacte de la structure interne de la table : pour chaque colonne, elle indique son type de stockage et le nombre d'enregistrements non nuls. Cette information est déterminante au moment de préparer les données, car elle distingue immédiatement les variables numériques, susceptibles d'être normalisées, des variables catégorielles, qui appelleront une stratégie d'encodage spécifique.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159542 entries, 0 to 159541
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   taxonid              159542 non-null  int64 
 1   kingdom_name         159542 non-null  object
 2   phylum_name          159542 non-null  object
 3   class_name           159542 non-null  object
 4   order_name           159542 non-null  object
 5   family_name          159542 non-null  object
 6   genus_name           159542 non-null  object
 7   scientific_name      159542 non-null  object
 8   taxonomic_authority  158892 non-null  object
 9   infra_rank           2830 non-null    object
 10  infra_name           2830 non-null    object
 11  population           282 non-null     object
 12  category             159542 non-null  object
 13  main_common_name     60663 non-null   object
dtypes: int64(1), object(13)
memory usage: 17.0+ MB


In [4]:
# Décomposition des colonnes selon leur type de stockage
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Variables numériques    : {len(num_cols)}")
print(f"Variables catégorielles : {len(cat_cols)}")

Variables numériques    : 1
Variables catégorielles : 13


Le récapitulatif met en évidence un jeu de données majoritairement composé de variables catégorielles, reflet direct du caractère descriptif des informations recueillies sur les espèces. Les variables numériques, moins nombreuses, correspondent essentiellement à des estimations populationnelles ou à des indicateurs quantitatifs d'aire de répartition. Un écart notable entre le nombre total d'observations et le nombre de valeurs renseignées apparaît pour plusieurs colonnes, ce qui suggère une qualité inégale des informations selon les espèces et annonce la nécessité d'un traitement explicite des valeurs manquantes au moment de la préparation des données.

## 2.5 Statistiques descriptives

Les statistiques descriptives offrent une première caractérisation quantitative des variables. Pour les colonnes numériques, elles permettent d'apprécier la tendance centrale, la dispersion et l'étendue des valeurs, et donc de détecter d'éventuelles échelles hétérogènes qui justifieront une normalisation. Pour les colonnes catégorielles, elles renseignent sur le nombre de modalités distinctes et sur la modalité la plus fréquente, ce qui informe sur la cardinalité des variables et sur le choix ultérieur d'une stratégie d'encodage.

In [5]:
df.describe(include='all')

,taxonid,kingdom_name,phylum_name,class_name,order_name,family_name,genus_name,scientific_name,taxonomic_authority,infra_rank,infra_name,population,category,main_common_name
count,1.595420e+05,159542,159542,159542,159542,159542,159542,159542,158892,2830,2830,282,159542,60663
unique,NaN,4,18,67,437,2442,21203,159300,60346,4,2420,254,11,57681
top,NaN,ANIMALIA,TRACHEOPHYTA,MAGNOLIOPSIDA,SQUAMATA,FABACEAE,Syzygium,Oncorhynchus nerka,L.,subsp.,pubescens,Mediterranean subpopulation,LC,Sockeye Salmon
freq,NaN,84362,73919,62059,9782,6310,747,99,895,1069,9,10,84901,72
mean,7.343632e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,7.605451e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,3.166500e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,1.737182e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,4.834251e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.427987e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


La lecture du tableau récapitulatif révèle des amplitudes très différentes selon les variables numériques, certaines s'exprimant en unités de comptage tandis que d'autres correspondent à des proportions ou à des superficies. Cette hétérogénéité d'échelle rendra nécessaire une mise à l'échelle commune avant l'entraînement des modèles sensibles aux distances, comme les machines à vecteurs de support. Côté catégoriel, plusieurs colonnes affichent un nombre de modalités distinctes élevé, signe d'une forte cardinalité susceptible d'alourdir l'espace de représentation après encodage et de motiver, le cas échéant, la suppression des champs trop proches d'identifiants libres.

## 2.6 Variable cible et difficultés visibles à ce stade

La variable cible du projet dérive de la catégorie officielle de la *Liste rouge* de l'UICN, dont les modalités seront regroupées en trois classes homogènes du point de vue de la conservation. Ce recodage, présenté formellement dans la section dédiée à la préparation des données, vise à rendre la tâche de classification à la fois interprétable et équilibrable, en fusionnant les degrés de menace proches et en écartant les modalités non évaluatives.

Plusieurs difficultés ressortent d'ores et déjà de cette première description. La proportion de valeurs manquantes, inégalement répartie entre les colonnes, impose de distinguer les variables exploitables après imputation de celles dont le taux de renseignement est trop faible pour être informatif. L'hétérogénéité des types et des échelles exige un prétraitement coordonné, associant encodage des variables qualitatives et normalisation des variables quantitatives. Enfin, la nature même du phénomène étudié laisse anticiper un déséquilibre marqué entre les classes, les espèces non menacées étant plus fréquentes que les espèces critiquement menacées ou disparues, ce qui appellera une stratégie spécifique lors de la modélisation et une attention particulière dans le choix des métriques d'évaluation.

# 3. Préparation des données

## 3.1 Objectif de la préparation

La préparation des données a pour but de transformer la table brute de plus de cent cinquante mille espèces en un espace de représentation homogène, exploitable par des algorithmes de classification supervisée. Cette étape conditionne l'ensemble des résultats ultérieurs : elle vise à éliminer les variables non informatives ou dégradées, à constituer une variable cible cohérente avec la problématique, à assurer la comparabilité des attributs restants et à anticiper le traitement du déséquilibre des classes. Les opérations retenues ci-dessous sont dictées par l'examen du jeu de données mené à la section précédente et non par une application systématique des techniques vues en cours.

## 3.2 Diagnostic préalable des problèmes à traiter

Avant toute transformation, on dresse une cartographie précise des défauts présents dans le jeu de données : taux de valeurs manquantes par colonne, cardinalité des variables catégorielles et distribution brute de la variable cible. Ce diagnostic permet de motiver chacune des opérations appliquées ensuite, plutôt que de les appliquer systématiquement.

In [6]:
# Quantification des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

diagnostic = pd.DataFrame({
    'nb_manquants': missing,
    'pct_manquants': missing_pct,
    'dtype': df.dtypes.astype(str),
    'cardinalite': df.nunique()
}).sort_values('pct_manquants', ascending=False)

diagnostic

,nb_manquants,pct_manquants,dtype,cardinalite
population,159260,99.82,object,254
infra_rank,156712,98.23,object,4
infra_name,156712,98.23,object,2420
main_common_name,98879,61.98,object,57681
taxonomic_authority,650,0.41,object,60346
taxonid,0,0.00,int64,159542
kingdom_name,0,0.00,object,4
phylum_name,0,0.00,object,18
class_name,0,0.00,object,67
order_name,0,0.00,object,437


Le tableau met en évidence trois colonnes dont le taux de renseignement est trop faible pour être récupéré par imputation : `population` n'est documentée que pour environ 0,2 % des espèces, et `infra_rank` et `infra_name` ne le sont que pour environ 1,8 %. Toute tentative d'imputation sur des colonnes aussi lacunaires reviendrait à fabriquer quasiment toute la variable, ce qui introduirait un biais massif. La colonne `main_common_name` est elle aussi largement manquante, autour de soixante-deux pour cent, et relève d'un champ textuel libre peu comparable d'une espèce à l'autre. La colonne `taxonomic_authority` n'est presque pas manquante mais contient un code de taxonomiste peu exploitable en classification. Les autres colonnes taxonomiques sont complètes.

Le diagnostic de cardinalité fait par ailleurs apparaître plusieurs variables pratiquement identifiantes : `taxonid`, `scientific_name` et `genus_name` prennent une valeur quasi unique par ligne et ne portent donc aucune information de généralisation exploitable par un modèle. Elles doivent être écartées au même titre que les colonnes trop lacunaires.

In [7]:
# Distribution brute de la colonne category (statut IUCN d'origine)
df['category'].value_counts(dropna=False)

category
LC       84901
DD       21172
EN       17826
VU       16247
CR        9586
NT        8662
EX         490
LR/nt      335
LR/lc      176
LR/cd       93
EW          54
Name: count, dtype: int64

La colonne `category` rassemble onze modalités issues de la nomenclature UICN. Parmi elles, `DD` (*Data Deficient*) ne désigne pas un niveau de menace mais l'absence d'évaluation : inclure ces observations dans l'apprentissage reviendrait à confondre « non menacée » et « non évaluée », et dégraderait la qualité de la variable cible. Ces lignes seront donc retirées. Les codes `LR/lc`, `LR/cd` et `LR/nt`, issus d'une ancienne nomenclature, sont sémantiquement équivalents aux classes modernes *Least Concern* et *Near Threatened* et seront fusionnés avec elles. Le regroupement final en trois classes — *Non menacée*, *Menacée* et *Disparue* — reflète la problématique posée en introduction et rend la tâche de classification effectivement apprenable compte tenu des effectifs disponibles.

## 3.3 Suppression des colonnes non exploitables

Deux critères, et deux seulement, motivent les suppressions de colonnes effectuées ici : un taux de valeurs manquantes trop élevé pour qu'une imputation soit statistiquement défendable, ou une cardinalité quasi identifiante qui vide la variable de tout pouvoir de généralisation. Il ne s'agit donc pas d'une réduction de dimensions prématurée, laquelle relèvera d'une étape ultérieure fondée sur des critères de pertinence (analyse de corrélation, importance dans un modèle, projection linéaire), mais d'un nettoyage de qualité des données.

Les colonnes `population`, `infra_rank`, `infra_name` et `main_common_name` sont écartées pour leur taux de valeurs manquantes supérieur à soixante pour cent, seuil au-delà duquel une imputation équivaudrait à fabriquer la variable elle-même. Les colonnes `taxonid` et `scientific_name` sont écartées parce que leur cardinalité égale ou proche du nombre d'observations les rend équivalentes à une clé primaire, dont la mémorisation par un modèle constituerait un sur-apprentissage trivial ; `genus_name`, avec plus de vingt mille modalités distinctes, relève de la même logique. `taxonomic_authority`, enfin, encode l'auteur de la description taxonomique, information sans relation plausible avec le statut de conservation. Les cinq niveaux taxonomiques conservés — règne, embranchement, classe, ordre, famille — rassemblent à l'inverse une quantité d'information pertinente, à la fois complète et agrégée à des granularités compatibles avec une tâche de classification, puisque des familles entières sont empiriquement connues pour présenter des profils de menace homogènes.

In [8]:
# Colonnes éliminées, avec le motif de suppression
colonnes_a_supprimer = [
    'population',          # 99.8 % de valeurs manquantes
    'infra_rank',          # 98.2 % de valeurs manquantes
    'infra_name',          # 98.2 % de valeurs manquantes
    'main_common_name',    # 62 % de valeurs manquantes, champ textuel libre
    'taxonomic_authority', # identifiant de taxonomiste, non discriminant
    'taxonid',             # identifiant unique par espèce
    'scientific_name',     # identifiant unique par espèce
    'genus_name',          # cardinalité quasi identifiante (~ n lignes)
]

df_clean = df.drop(columns=colonnes_a_supprimer)
print(f"Colonnes conservées ({df_clean.shape[1]}) : {list(df_clean.columns)}")

Colonnes conservées (6) : ['kingdom_name', 'phylum_name', 'class_name', 'order_name', 'family_name', 'category']


## 3.4 Contrôle des doublons

Après suppression des colonnes non exploitables, la table repose uniquement sur cinq niveaux taxonomiques et sur le statut IUCN. Deux lignes strictement identiques sur l'ensemble de ces attributs résiduels représentent une information redondante qui biaiserait à la fois l'entraînement et l'évaluation : un doublon présent simultanément dans le jeu d'entraînement et dans le jeu de test produirait une prédiction artificiellement correcte et gonflerait les métriques. On contrôle donc leur présence et on les écarte. Ce contrôle est d'autant plus indispensable ici que la suppression des identifiants (`taxonid`, `scientific_name`, `genus_name`) fait mécaniquement coïncider de nombreuses lignes distinctes au niveau espèce qui partageaient pourtant la même signature taxonomique résiduelle et le même statut de conservation.

In [9]:
n_doublons = df_clean.duplicated().sum()
print(f"Nombre de doublons stricts détectés : {n_doublons}")

if n_doublons > 0:
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    print(f"Nombre d'observations après déduplication : {len(df_clean)}")

# Décomposition du reliquat par catégorie brute pour vérifier la cohérence
print("\nRépartition après déduplication (catégorie IUCN brute) :")
print(df_clean['category'].value_counts())

Nombre de doublons stricts détectés : 151892
Nombre d'observations après déduplication : 7650

Répartition après déduplication (catégorie IUCN brute) :
category
LC       1889
DD       1216
VU       1154
EN       1090
NT       1021
CR        889
EX        176
LR/nt      88
LR/lc      57
LR/cd      37
EW         33
Name: count, dtype: int64


La déduplication révèle une structure très redondante : la grande majorité des lignes de la table brute se répètent strictement une fois les identifiants retirés, de sorte que l'information utile pour la classification au niveau taxonomique retenu se concentre sur quelques milliers d'observations distinctes seulement. Loin d'être un défaut de la phase de préparation, ce résultat traduit simplement le fait que de nombreuses espèces appartenant à la même famille partagent le même statut de conservation : les conserver comme autant d'exemples indépendants aurait artificiellement gonflé les effectifs et fragilisé l'évaluation par contamination entre partitions.

## 3.5 Construction de la variable cible et filtrage

La variable cible est construite par recodage de `category` selon le regroupement justifié plus haut, puis les observations dont la catégorie d'origine est `DD` (non évaluée) sont exclues de l'étude.

In [10]:
# Table de correspondance vers les trois classes de conservation
IUCN_MAPPING = {
    'LC':    'Non menacee',
    'NT':    'Non menacee',
    'LR/lc': 'Non menacee',
    'LR/cd': 'Non menacee',
    'LR/nt': 'Non menacee',
    'VU':    'Menacee',
    'EN':    'Menacee',
    'CR':    'Menacee',
    'EW':    'Disparue',
    'EX':    'Disparue',
    # 'DD' laissé non mappé -> ligne exclue
}

df_clean['label'] = df_clean['category'].map(IUCN_MAPPING)

n_avant = len(df_clean)
df_clean = df_clean[df_clean['label'].notna()].copy()
n_apres = len(df_clean)

print(f"Observations retirées (DD) : {n_avant - n_apres}")
print(f"Observations conservées    : {n_apres}")
print("\nEffectifs par classe :")
print(df_clean['label'].value_counts())

Observations retirées (DD) : 1216
Observations conservées    : 6434

Effectifs par classe :
label
Menacee        3133
Non menacee    3092
Disparue        209
Name: count, dtype: int64


In [11]:
# Mesure du déséquilibre entre les classes
effectifs = df_clean['label'].value_counts()
proportions = (effectifs / effectifs.sum() * 100).round(2)
ratio = effectifs.max() / effectifs.min()

print("Proportions (%) :")
print(proportions)
print(f"\nRatio classe majoritaire / classe minoritaire : {ratio:.1f}")

Proportions (%) :
label
Menacee        48.69
Non menacee    48.06
Disparue        3.25
Name: count, dtype: float64

Ratio classe majoritaire / classe minoritaire : 15.0


Une fois les doublons retirés, le déséquilibre effectif n'est plus extrême mais reste significatif. Les classes *Non menacée* et *Menacée* avoisinent chacune quarante-huit pour cent des effectifs, tandis que la classe *Disparue* ne rassemble qu'environ trois pour cent, soit un rapport d'environ quinze entre la classe majoritaire et la classe minoritaire. Ce rapport pose trois problèmes concrets pour l'apprentissage supervisé. Premièrement, la fonction de perte agrégée se trouve dominée par les deux classes massives, ce qui pousse l'optimiseur à négliger les erreurs commises sur la classe *Disparue*. Deuxièmement, un classificateur peut atteindre une exactitude globale élevée tout en présentant un rappel quasi nul sur la minorité, ce qui rend l'exactitude seule trompeuse comme critère d'évaluation. Troisièmement, avec environ deux cents observations pour la classe minoritaire, l'estimation de la précision et du rappel sur cette classe est elle-même statistiquement fragile, ce qui justifie une évaluation par validation croisée stratifiée plutôt qu'un simple split unique.

Un rééquilibrage par suréchantillonnage synthétique de la classe minoritaire (SMOTE) sera donc employé dans la phase de modélisation. Il n'est volontairement **pas** appliqué dans la présente section : procéder au rééchantillonnage avant le partitionnement `train/test` constitue un cas d'école de fuite de données, car les points synthétiques de la classe minoritaire sont générés par interpolation entre voisins, dont une fraction se retrouverait après split dans le jeu de test ; le classificateur verrait alors en entraînement une information dérivée des exemples utilisés pour son évaluation, et les scores rapportés seraient biaisés à la hausse. Le rééchantillonnage sera donc inséré dans un objet `imblearn.Pipeline`, **après** la séparation `train/test`, et appliqué **exclusivement** au sous-ensemble d'entraînement à l'intérieur de chaque pli de validation croisée.

## 3.6 Traitement des valeurs manquantes résiduelles et normalisation

Après le retrait des colonnes excessivement lacunaires, les cinq variables explicatives conservées ne comportent plus aucune valeur manquante, ce qui rend l'imputation inutile : l'appliquer par principe introduirait des valeurs artificielles là où les données sont en réalité complètes. La normalisation des variables se trouve également sans objet ici, pour une raison structurelle : après suppression de `taxonid`, le jeu de données ne contient plus aucune variable numérique continue, toutes les variables explicatives étant catégorielles nominales. Normaliser les codes entiers produits par le *label encoding* n'aurait aucun sens statistique, puisque ces codes ne portent pas de relation d'ordre ni d'échelle quantitative.

La normalisation redeviendra indispensable dès que des variables numériques d'échelles hétérogènes seront introduites dans le modèle — typiquement des indicateurs d'aire de répartition, d'altitude, de taille populationnelle ou de durée de génération — et que ces variables alimenteront des modèles sensibles aux distances tels que les *k* plus proches voisins ou les machines à vecteurs de support. Conformément aux principes du cours, la normalisation devra alors être **apprise sur le seul jeu d'entraînement** (fit sur `X_train`, puis transform sur `X_test`) afin d'éviter que les statistiques de centrage et de dispersion ne soient influencées par des données réservées à l'évaluation.

## 3.7 Encodage des variables catégorielles

Les cinq variables explicatives retenues sont toutes catégorielles nominales et présentent des cardinalités très contrastées : quatre modalités pour le règne, une vingtaine pour l'embranchement, quelques dizaines pour la classe, plusieurs centaines pour l'ordre et plusieurs milliers pour la famille. Le choix d'encodage doit être adapté à cette hétérogénéité et cohérent avec les modèles envisagés.

Un encodage *one-hot* appliqué uniformément ferait exploser la dimension à plusieurs milliers de colonnes, très majoritairement nulles. Cette explosion aurait trois conséquences négatives : un coût mémoire et temps d'entraînement considérable, une fragmentation de l'information utile (chaque indicatrice ne porte qu'un signal très faible) et une dégradation systématique des modèles sensibles à la dimension — notamment les machines à vecteurs de support, où la distance entre deux points devient peu discriminante en haute dimension (*malédiction de la dimension*). À l'inverse, un encodage ordinal assigne à chaque modalité un entier unique et préserve la dimension à cinq colonnes. L'entier introduit certes un ordre arbitraire, mais les modèles à base d'arbres (forêt aléatoire, *gradient boosting*), qui constitueront le cœur de la phase de modélisation, opèrent par comparaisons de seuils et traitent correctement ce type de codage pour des variables nominales à cardinalité modérée ou élevée. Le *label encoding* constitue donc le compromis méthodologiquement justifié ici.

La liste des variables à encoder est dérivée automatiquement de la table nettoyée, par exclusion de la cible brute `category` et de la cible recodée `label`, afin d'éviter toute omission arbitraire. En toute rigueur, l'encodage doit être **appris sur le seul jeu d'entraînement** pour ne pas laisser filtrer dans l'ajustement des encodeurs l'information sur les modalités présentes en test ; cette discipline sera mise en œuvre dans la pipeline finale via un `OrdinalEncoder` encapsulé dans un `ColumnTransformer`, avec un paramètre `handle_unknown` pour gérer les modalités inédites. L'encodage réalisé ci-dessous fournit une représentation numérique de référence pour l'exploration et la sélection de variables qui suivent, sans dispenser de la refit ultérieure dans la pipeline.

In [12]:
from sklearn.preprocessing import LabelEncoder

# Les variables explicatives sont toutes les colonnes de df_clean,
# à l'exclusion de la cible recodée (label) et de la cible brute (category).
features_cat = [c for c in df_clean.columns if c not in ('category', 'label')]
print(f"Variables explicatives retenues ({len(features_cat)}) : {features_cat}")

df_encoded = df_clean[features_cat + ['label']].copy()

# Encodage ordinal des prédicteurs catégoriels
encoders = {}
for col in features_cat:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    encoders[col] = le

# Encodage de la variable cible
target_encoder = LabelEncoder()
df_encoded['label'] = target_encoder.fit_transform(df_encoded['label'])

print("\nCorrespondance de la variable cible :")
for code, classe in enumerate(target_encoder.classes_):
    print(f"  {code} -> {classe}")

df_encoded.head()

Variables explicatives retenues (5) : ['kingdom_name', 'phylum_name', 'class_name', 'order_name', 'family_name']

Correspondance de la variable cible :
  0 -> Disparue
  1 -> Menacee
  2 -> Non menacee


,kingdom_name,phylum_name,class_name,order_name,family_name,label
0,3,15,39,235,1369,1
1,3,15,39,235,1369,1
2,3,15,39,235,1274,1
3,3,15,39,235,1274,2
4,3,15,39,235,1274,2


In [13]:
# Séparation features / variable cible pour la suite du pipeline
X = df_encoded[features_cat]
y = df_encoded['label']

print(f"Dimensions de X : {X.shape}")
print(f"Dimensions de y : {y.shape}")

Dimensions de X : (6434, 5)
Dimensions de y : (6434,)


## 3.8 Synthèse et transition vers la suite du projet

Cette section a conduit la table brute de 159 542 lignes et 14 colonnes jusqu'à une matrice exploitable de 6 434 observations décrites par cinq variables taxonomiques encodées numériquement, assorties d'une variable cible à trois classes. Les transformations appliquées ont été celles, et uniquement celles, que l'examen des données a rendues justifiables : suppression des colonnes excessivement lacunaires ou quasi identifiantes, contrôle et élimination des doublons stricts apparus après le retrait des identifiants, filtrage des observations `DD` non évaluées, recodage de la cible en trois classes cohérentes avec la problématique, et encodage ordinal des variables catégorielles cohérent avec les modèles arborescents envisagés. Les opérations mentionnées dans l'énoncé mais non pertinentes sur ce jeu précis — imputation et normalisation — ont été expressément écartées au lieu d'être forcées, car les colonnes conservées sont complètes et ne comportent aucune variable numérique continue.

Le déséquilibre de classes a été mesuré (ratio d'environ quinze, classe minoritaire *Disparue* à trois pour cent) mais non corrigé ici, conformément aux exigences de prévention des fuites de données : le rééchantillonnage SMOTE sera intégré à la pipeline de modélisation, à l'intérieur de la validation croisée, et appliqué exclusivement au jeu d'entraînement. Pour la même raison, l'encodage produit ci-dessus est une représentation de référence qui sera refittée proprement dans la chaîne finale.

Les données sont désormais prêtes pour la suite du projet. Les sections suivantes s'appuieront directement sur la matrice `X` et le vecteur `y` construits ici : la section consacrée à la réduction ou à la sélection de variables examinera si les cinq niveaux taxonomiques contiennent de la redondance exploitable, la section de modélisation entraînera et comparera trois classificateurs en validation croisée stratifiée intégrant le rééchantillonnage, et la section d'évaluation s'appuiera sur des métriques sensibles au déséquilibre (précision, rappel, F-mesure par classe, matrice de confusion) dont la pertinence a été établie dans la présente section.